# Artists Table Cleaning

**Purpose:** Stores one record per artist employed by the company.

**Expected grain:** One row per artist.

**Primary key:** `artist_id`

**Important checks:**
- `artist_id` should be unique and non-null.
- `email` should follow the approved company format.
- `weekly_capacity_hours` should not be negative.
- `active_flag` should contain only `Y` or `N`.
- Missing manager IDs may be valid for department heads.

## Cleaning decisions

- Preserve valid null `manager_id` values for department heads.
- Standardize `active_flag` to `Y` or `N`.
- Treat negative weekly capacity as invalid and investigate before correcting it.
- Fill missing email addresses only when the approved company naming convention
  can be applied confidently.
- Do not silently remove records with unresolved business-rule problems.

In [657]:
# Connect to this notebook's private project snapshot
from pathlib import Path
import os
import duckdb

project_root = Path(os.environ["INSIGHT_PROJECT_ROOT"])
database_path = Path(
    os.environ["INSIGHT_NOTEBOOK_DATABASE"]
)
connection = duckdb.connect(str(database_path))
for folder_name in ('data', 'raw', 'clean', 'public'):
    connection.execute(
        f'CREATE SCHEMA IF NOT EXISTS "{folder_name}"'
    )
connection.execute(
    "SET search_path = 'raw,clean,public,data,main'"
)
try:
    connection.execute("LOAD inflector")
    inflector_available = True
except Exception:
    inflector_available = False
project_tables = connection.execute(
    """
    SELECT
        table_schema AS folder_name,
        table_name,
        table_schema || '.' || table_name AS sql_reference
    FROM information_schema.tables
    WHERE table_schema NOT IN (
        'main',
        'temp',
        'information_schema',
        'pg_catalog'
    )
      AND table_name NOT LIKE '_insight_%'
    ORDER BY table_schema, table_name
    """
).fetchall()
if project_tables:
    print('Project tables:')
    for folder_name, table_name, sql_reference in project_tables:
        print(f'  {sql_reference}')
else:
    print('No project tables are currently available.')

Project tables:
  clean.raw_artists_cleaned
  raw.raw_artists
  raw.raw_clients
  raw.raw_projects
  raw.raw_reviews
  raw.raw_shots
  raw.raw_time_entries
  staging.artists

In [658]:
%%sql
DESCRIBE raw.raw_artists;

SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT artist_id) AS possible_duplicate_ids,
    COUNT(artist_id) AS populated_artist_ids,
    COUNT(DISTINCT artist_id) AS unique_artist_ids
FROM raw_artists;

total_rows,possible_duplicate_ids,populated_artist_ids,unique_artist_ids
62,2,62,60


In [659]:
%%sql
SELECT
    'Missing artist ID' AS issue,
    COUNT(*) AS affected_rows
FROM raw_artists
WHERE artist_id IS NULL

UNION ALL

SELECT
    'Duplicate artist ID',
    COUNT(*)
FROM (
    SELECT artist_id
    FROM raw_artists
    GROUP BY artist_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT
    'Missing email',
    COUNT(*)
FROM raw_artists
WHERE email IS NULL
   OR TRIM(email) = ''

UNION ALL

SELECT
    'Missing hourly cost',
    COUNT(*)
FROM raw_artists
WHERE hourly_cost_usd IS NULL

UNION ALL

SELECT
    'Negative or Missing weekly capacity',
    COUNT(*)
FROM raw_artists
WHERE CAST(LEFT(weekly_capacity_hours, 2) AS INT)  < 0 OR weekly_capacity_hours IS NULL

UNION ALL


SELECT
    'Unexpected active flag',
    COUNT(*)
FROM raw_artists
WHERE UPPER(TRIM(active_flag)) NOT IN ('Y', 'N')
   OR active_flag IS NULL;

issue,affected_rows
Missing artist ID,0
Duplicate artist ID,2
Missing email,1
Missing hourly cost,1
Negative or Missing weekly capacity,1
Unexpected active flag,3


In [660]:
%%sql
CREATE SCHEMA IF NOT EXISTS staging;

CREATE OR REPLACE TABLE staging.artists AS
SELECT DISTINCT
    artist_id,
    TRIM(artist_name) AS artist_name,
    
    CASE
        WHEN UPPER(TRIM(department)) = 'FX'
            THEN 'FX'
         WHEN UPPER(TRIM(department)) = 'ROTO/PAINT'
             THEN 'Roto/Paint'
        ELSE inflector_to_title_case(TRIM(department))
    END AS department,
        
    TRIM(role) AS role,
    TRIM(seniority) AS seniority,
    TRIM(location) AS location,
    LEFT(weekly_capacity_hours, 2) AS weekly_capacity_hours,
    hourly_cost_usd,
    hire_date,
    manager_id,
    LOWER(TRIM(email)) AS email,

    CASE
        WHEN UPPER(TRIM(active_flag)) IN ('Y', 'YES', 'TRUE', 'ACTIVE')
            THEN 'Y'
        WHEN UPPER(TRIM(active_flag)) IN ('N', 'NO', 'FALSE', 'INACTIVE')
            THEN 'N'
        ELSE NULL
    END AS active_flag

FROM raw.raw_artists;


SELECT *
FROM staging.artists

artist_id,artist_name,department,role,seniority,location,weekly_capacity_hours,hourly_cost_usd,hire_date,manager_id,email,active_flag
ART-037,Parker Scott,Assets,Modeler,Mid,Remote-US,24,63,2019-01-24,ART-047,parker.scott@spectraforgevfx.example,Y
ART-001,Morgan Jackson,Assets,Modeler,Junior,Vancouver,40,54,2019-01-19,ART-047,morgan.jackson@spectraforgevfx.example,Y
ART-010,Morgan Patel,Compositing,Senior Compositor,Senior,Los Angeles,35,100,2021-07-25,ART-060,morgan.patel@spectraforgevfx.example,Y
ART-053,Sofia Young,FX,Senior FX Artist,Senior,London,40,107,2022-08-17,ART-058,sofia.young@spectraforgevfx.example,Y
ART-032,Elena Walker,Lighting,Senior Lighting Artist,Senior,Montreal,40,103,2021-09-26,ART-014,elena.walker@spectraforgevfx.example,N
ART-015,Parker Thomas,Compositing,Senior Compositor,Senior,Vancouver,24,88,2021-03-17,ART-010,parker.thomas@spectraforgevfx.example,Y
ART-007,Alex Anderson,Matte Painting,Matte Painter,Mid,London,40,68,2022-08-23,NULL,alex.anderson@spectraforgevfx.example,Y
ART-059,Drew Patel,Compositing,Compositor,Mid,Atlanta,40,58,2024-08-12,ART-025,drew.patel@spectraforgevfx.example,Y
ART-014,Nico Harris,Lighting,Senior Lighting Artist,Senior,Vancouver,40,86,2020-10-07,ART-023,nico.harris@spectraforgevfx.example,N
ART-055,Blake Garcia,Matte Painting,Matte Painter,Mid,Montreal,40,68,2025-08-21,NULL,blake.garcia@spectraforgevfx.example,Y


## Preliminary Standardization

I branched off a protective staging version of the raw_artists dataset and applied preliminary standardization to the following rows:
    
    * Two Duplicate rows were confirmed to be a data error and were removed, preserving only one row for each artist.
    * The department column was standardized into Title Case and contains only approved categories.
    * Weekly capacity hours was stripped down to only the weekly capacity in integers.
    * The active flag column was standardized to only accept "Y" for active and "N" for inactive.

In [661]:
%%sql
SELECT artist_id, weekly_capacity_hours
FROM staging.artists
WHERE CAST(weekly_capacity_hours AS INT) > 40;

UPDATE staging.artists
SET weekly_capacity_hours = 40
WHERE CAST(weekly_capacity_hours AS INT) > 40 AND artist_id = 'ART-033';

SELECT artist_id, weekly_capacity_hours
FROM staging.artists
WHERE artist_id = 'ART-033';

artist_id,weekly_capacity_hours
ART-033,40


## Fixed weekly_capacity_hours greater than 40

ART-033 had a record of 80 for weekly_capacity_hours. After talking with the Head of Production they confirmed that this is an error and ART-033 is a normal
full time employee with a weekly hour capacity as 40.

In [662]:
%%sql
WITH null_list AS (
SELECT *
FROM staging.artists
WHERE 
         artist_id IS NULL
   OR artist_name IS NULL
   OR department IS NULL
   OR role IS NULL
   OR seniority IS NULL
   OR location IS NULL
   OR weekly_capacity_hours IS NULL
   OR hourly_cost_usd IS NULL
   OR hire_date IS NULL
   OR active_flag IS NULL
   OR manager_id IS NULL
   OR email IS NULL
  )
  
  SELECT *
  FROM null_list;

artist_id,artist_name,department,role,seniority,location,weekly_capacity_hours,hourly_cost_usd,hire_date,manager_id,email,active_flag
ART-007,Alex Anderson,Matte Painting,Matte Painter,Mid,London,40,68,2022-08-23,NULL,alex.anderson@spectraforgevfx.example,Y
ART-055,Blake Garcia,Matte Painting,Matte Painter,Mid,Montreal,40,68,2025-08-21,NULL,blake.garcia@spectraforgevfx.example,Y
ART-029,Elena Lewis,Matte Painting,Matte Painter,Mid,Montreal,40,66,2022-08-28,NULL,elena.lewis@spectraforgevfx.example,Y
ART-008,Casey Martinez,Compositing,Compositor,Mid,Atlanta,35,NULL,2023-05-01,ART-019,casey.martinez@spectraforgevfx.example,Y
ART-051,Casey Hall,Matte Painting,Matte Painter,Mid,Remote-US,40,65,2024-03-27,NULL,casey.hall@spectraforgevfx.example,Y
ART-006,Finley Martin,Animation,Senior Animator,Senior,Remote-US,24,85,2021-11-25,NULL,NULL,Y


In [663]:
%%sql
UPDATE staging.artists
SET hourly_cost_usd = 58
WHERE artist_id = 'ART-008';

SELECT *
FROM staging.artists AS s
WHERE s.artist_id = 'ART-008';

artist_id,artist_name,department,role,seniority,location,weekly_capacity_hours,hourly_cost_usd,hire_date,manager_id,email,active_flag
ART-008,Casey Martinez,Compositing,Compositor,Mid,Atlanta,35,58,2023-05-01,ART-019,casey.martinez@spectraforgevfx.example,Y


In [664]:
%%sql
UPDATE staging.artists
SET email = 'finley.martin@spectraforgevfx.example'
WHERE artist_id = 'ART-006';

SELECT *
FROM staging.artists AS s
WHERE s.artist_id = 'ART-006';

artist_id,artist_name,department,role,seniority,location,weekly_capacity_hours,hourly_cost_usd,hire_date,manager_id,email,active_flag
ART-006,Finley Martin,Animation,Senior Animator,Senior,Remote-US,24,85,2021-11-25,NULL,finley.martin@spectraforgevfx.example,Y


## Fixed Nulls

Two rows had missing values in the dataset. ART-008 was missing an hourly_cost_usd record and ART-006 was missing an email record. These values were asked to and 
relayed from the Head of Production at Spectraforge VFX. Null manager_ids were confirmed to be correct and signal that an artist has no manager.

In [665]:
%%sql
SELECT artist_id, weekly_capacity_hours
FROM staging.artists
WHERE CAST(weekly_capacity_hours AS INT) < 0;

UPDATE staging.artists
SET weekly_capacity_hours = 32
WHERE artist_id = 'ART-040';

SELECT *
FROM staging.artists
WHERE artist_id = 'ART-040';

artist_id,artist_name,department,role,seniority,location,weekly_capacity_hours,hourly_cost_usd,hire_date,manager_id,email,active_flag
ART-040,Noah Wilson,Compositing,Compositor,Junior,Atlanta,32,49,2022-06-24,ART-060,noah.wilson@spectraforgevfx.example,Y


## Fixed Negative weekly_capacity_hours

Artists ART-040 had a weekly_capacity_hours record that was -8, this record was confirmed to be incorrect by the Head of Production and I was given the correct value for their
capacity which should be 32 hours.

In [666]:
%%sql
SELECT artist_id, hourly_cost_usd
FROM staging.artists
WHERE regexp_matches(hourly_cost_usd, '^\d+$') = false;

SELECT artist_id, regexp_replace(hourly_cost_usd, '[^0-9]', '', 'g') AS hourly_cost_usd
FROM staging.artists
WHERE artist_id IN ('ART-025', 'ART-058', 'ART-042');

UPDATE staging.artists
SET hourly_cost_usd = regexp_replace(hourly_cost_usd, '[^0-9]', '', 'g')
WHERE artist_id IN ('ART-025', 'ART-058', 'ART-042');

SELECT *
FROM staging.artists
WHERE artist_id IN ('ART-025', 'ART-058', 'ART-042');

artist_id,artist_name,department,role,seniority,location,weekly_capacity_hours,hourly_cost_usd,hire_date,manager_id,email,active_flag
ART-025,Maya Lee,Compositing,Senior Compositor,Senior,Los Angeles,40,110,2024-07-19,ART-024,maya.lee@spectraforgevfx.example,Y
ART-042,Sam Anderson,FX,FX Artist,Junior,Los Angeles,40,95,2019-11-12,ART-058,sam.anderson@spectraforgevfx.example,Y
ART-058,Avery Allen,FX,Senior FX Artist,Senior,Los Angeles,0,45,2024-07-02,ART-053,avery.allen@spectraforgevfx.example,Y


In [667]:
SELECT email, COUNT(*)
FROM staging.artists
GROUP BY email
HAVING COUNT(*) > 1;

SELECT *
FROM staging.artists
WHERE email IN ('nico.patel@spectraforgevfx.example', 'liam.young@spectraforgevfx.example');

DELETE FROM staging.artists
WHERE artist_id IN ('ART-052', 'ART-005');

UPDATE staging.artists
SET email = 'riley.allen@spectraforgevfx.example'
WHERE artist_id = 'ART-013' AND email = 'nico.patel@spectraforgevfx.example';

SELECT email, COUNT(*)
FROM staging.artists
GROUP BY email
HAVING COUNT(*) > 1;


email,count_star()


In [669]:
ALTER TABLE staging.artists
ALTER COLUMN weekly_capacity_hours TYPE INTEGER;

ALTER TABLE staging.artists
ALTER COLUMN hourly_cost_usd TYPE INTEGER;

CREATE SCHEMA IF NOT EXISTS clean;

CREATE OR REPLACE TABLE clean.raw_artists_cleaned AS
SELECT
    artist_id,
    artist_name,
    department,
    role,
    seniority,
    location,
    weekly_capacity_hours,
    hourly_cost_usd,
    hire_date,
    active_flag,
    manager_id,
    email
FROM staging.artists;

Count
58
